# Final Clean Model Pipeline

This notebook is only for the final clean safe non-proxy model pipeline.

It does **not** include side experiments like:
- skew correction
- outlier capping
- proxy-feature add-backs
- alternative model families

Those stay in the experimentation notebooks.

## Small Pipeline Steps

We will build this notebook box by box in this order:

1. Read the raw CSV file.
2. Filter to the clean safe non-proxy modeling rows and define the target.
3. Parse and engineer the core final features.
4. Create the grouped categorical features: `occupation_group` and `purpose_group`.
5. Handle missing values and build the final CatBoost-native feature frame.
6. Define the four final feature subsets.
7. Train and evaluate the final clean native `CatBoost` models.
8. Save the final model outputs for the report.

Current final-model target choice:
- clean safe non-proxy feature set
- native categorical `CatBoost`
- grouped categoricals kept as raw categories
- no skew correction

## Box 1: Read Raw CSV

This first box only reads a small debug subset of the raw LendingClub CSV directly. No cached-stage files are used in this notebook.

Later, once the pipeline is correct, we can switch this to the full CSV by removing the `nrows` limit.

In [1]:
from IPython.display import display
import pandas as pd
from pathlib import Path

CSV_PATH = Path(r"C:\Users\Vido\Desktop\EPL448\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv")
DEBUG_NROWS = 100_000
raw_df = pd.read_csv(CSV_PATH, low_memory=False, nrows=DEBUG_NROWS)

print("CSV_PATH:", CSV_PATH)
print("DEBUG_NROWS:", DEBUG_NROWS)
print("raw_df shape:", raw_df.shape)
raw_df.head()


CSV_PATH: C:\Users\Vido\Desktop\EPL448\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv
DEBUG_NROWS: 100000
raw_df shape: (100000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## Box 2: Filter Rows And Define Target

This box keeps only the rows used in the final clean modeling path.

It does only row filtering and target creation:
- keep only `Fully Paid` and `Charged Off`
- define `target` with `Charged Off = 1`
- keep only `Individual` applications
- keep only `home_ownership` in `MORTGAGE`, `RENT`, or `OWN`


In [2]:
model_df = raw_df.copy()

model_df = model_df[model_df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
model_df["target"] = (model_df["loan_status"] == "Charged Off").astype(int)

model_df = model_df[model_df["application_type"] == "Individual"].copy()
model_df = model_df[model_df["home_ownership"].isin(["MORTGAGE", "RENT", "OWN"])].copy()
model_df = model_df.reset_index(drop=True)
source_df = model_df.copy()

class_counts = model_df["target"].value_counts().sort_index()
class_percentages = model_df["target"].value_counts(normalize=True).sort_index().mul(100).round(2)
class_summary_df = pd.DataFrame({
    "class_label": ["Fully Paid (0)", "Charged Off (1)"],
    "count": [int(class_counts.get(0, 0)), int(class_counts.get(1, 0))],
    "percentage": [float(class_percentages.get(0, 0.0)), float(class_percentages.get(1, 0.0))],
})

print("model_df shape:", model_df.shape)
display(class_summary_df)
model_df[["loan_status", "target", "application_type", "home_ownership"]].head()


model_df shape: (87496, 152)


,class_label,count,percentage
0,Fully Paid (0),69990,79.99
1,Charged Off (1),17506,20.01


,loan_status,target,application_type,home_ownership
0,Fully Paid,0,Individual,MORTGAGE
1,Fully Paid,0,Individual,MORTGAGE
2,Fully Paid,0,Individual,MORTGAGE
3,Fully Paid,0,Individual,RENT
4,Fully Paid,0,Individual,MORTGAGE


## Box 3: Remove Unwanted Columns

This box removes the columns that are not part of the final clean safe non-proxy pipeline.

We remove four groups:
- future leakage columns
- joint-application-only columns
- obsolete or ID-style columns
- proxy-style lender decision columns

We also remove a few raw columns that will be replaced later during feature engineering.

In [3]:
FUTURE_COLUMNS = [
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status", "deferral_term",
    "hardship_amount", "hardship_start_date", "hardship_end_date", "payment_plan_start_date",
    "hardship_length", "hardship_dpd", "hardship_loan_status",
    "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
    "hardship_last_payment_amount", "disbursement_method", "debt_settlement_flag",
    "debt_settlement_flag_date", "settlement_status", "settlement_date", "settlement_amount",
    "settlement_percentage", "settlement_term", "collection_recovery_fee", "last_pymnt_amnt",
    "last_pymnt_d", "next_pymnt_d", "out_prncp", "out_prncp_inv", "policy_code",
    "recoveries", "total_pymnt", "total_pymnt_inv", "total_rec_int", "total_rec_late_fee",
    "total_rec_prncp", "pymnt_plan"
]

JOINT_COLUMNS = [
    "dti_joint", "annual_inc_joint", "verification_status_joint", "revol_bal_joint",
    "sec_app_fico_range_low", "sec_app_fico_range_high", "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths", "sec_app_mort_acc", "sec_app_open_acc", "sec_app_revol_util",
    "sec_app_open_act_il", "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med", "sec_app_mths_since_last_major_derog"
]

OBSOLETE_COLUMNS = [
    "id", "initial_list_status", "member_id", "bc_open_to_buy", "last_credit_pull_d",
    "desc", "funded_amnt_inv", "title", "url", "zip_code", "funded_amnt"
]

PROXY_COLUMNS = [
    "fico_range_low", "fico_range_high", "grade", "sub_grade", "installment", "int_rate"
]

RAW_COLUMNS_REPLACED_LATER = [
    "issue_d", "earliest_cr_line", "emp_title", "purpose", "addr_state"
]

columns_to_drop = FUTURE_COLUMNS + JOINT_COLUMNS + OBSOLETE_COLUMNS + PROXY_COLUMNS + RAW_COLUMNS_REPLACED_LATER
columns_to_drop = [column for column in columns_to_drop if column in model_df.columns]

model_df = model_df.drop(columns=columns_to_drop).copy()

print("Dropped columns:", len(columns_to_drop))
print("Remaining shape:", model_df.shape)
pd.DataFrame({"dropped_column": columns_to_drop}).head(20)


Dropped columns: 75
Remaining shape: (87496, 77)


,dropped_column
0,hardship_flag
1,hardship_type
2,hardship_reason
3,hardship_status
4,deferral_term
5,hardship_amount
6,hardship_start_date
7,hardship_end_date
8,payment_plan_start_date
9,hardship_length


## Box 4: Core Feature Engineering

This box creates the core engineered features used in the final clean model.

How each feature is built:

- `term`: extract the numeric part from the text value. For example, `36 months -> 36` and `60 months -> 60`.
- `emp_length`: extract the numeric part from the text value. For example, `10+ years -> 10`, `2 years -> 2`, and `< 1 year -> 1`. If no number is found, the value stays missing.
- `credit_maturity`: convert `issue_d` and `earliest_cr_line` to dates, then compute the difference in months:
  `credit_maturity = months(issue_d - earliest_cr_line)`
- `emp_length_cat`: bucket numeric `emp_length` into broad groups using these bins:
  `1-2 -> '1'`, `3-4 -> '2'`, `5-6 -> '3'`, `7-8 -> '4'`, `9-10 -> '5'`, and missing values become `unknown`.

So this box is purely mechanical preprocessing:
- pull numbers out of text fields
- convert date fields into a month-difference feature
- convert detailed employment length into a small categorical grouping


In [4]:
def parse_numeric_from_text(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.extract(r"(\d+)", expand=False), errors="coerce")

engineered_df = model_df.copy()

issue_dates = pd.to_datetime(source_df["issue_d"], format="%b-%Y", errors="coerce")
earliest_credit_dates = pd.to_datetime(source_df["earliest_cr_line"], format="%b-%Y", errors="coerce")

engineered_df["term"] = parse_numeric_from_text(source_df["term"])
engineered_df["emp_length"] = parse_numeric_from_text(source_df["emp_length"])

engineered_df["credit_maturity"] = (
    (issue_dates.dt.year - earliest_credit_dates.dt.year) * 12
    + (issue_dates.dt.month - earliest_credit_dates.dt.month)
)

engineered_df["emp_length_cat"] = pd.cut(
    engineered_df["emp_length"],
    bins=[0, 2, 4, 6, 8, 10],
    labels=["1", "2", "3", "4", "5"],
)
engineered_df["emp_length_cat"] = engineered_df["emp_length_cat"].cat.add_categories("unknown").fillna("unknown")

print("engineered_df shape:", engineered_df.shape)
engineered_df[["term", "emp_length", "credit_maturity", "emp_length_cat"]].head()


engineered_df shape: (87496, 79)


,term,emp_length,credit_maturity,emp_length_cat
0,36,10.0,148,5
1,36,10.0,192,5
2,60,3.0,210,2
3,36,4.0,338,2
4,36,10.0,306,5


## Box 5: Create Grouped Categorical Features

This box creates the two grouped categorical features used in the final clean model.

How `occupation_group` is built:
- start from raw `emp_title`
- convert it to lowercase text
- check for keywords that map the title into a broad occupation family
- if no keyword matches, assign `other`
- if the value is missing or empty, assign `unknown`

How `purpose_group` is built:
- start from raw `purpose`
- merge similar loan purposes into broader groups
- examples:
  - `credit_card` and `debt_consolidation` become `debt_refinancing`
  - `home_improvement`, `house`, and `moving` become `housing_related`
  - `car`, `major_purchase`, `vacation`, and `wedding` become `major_or_discretionary_purchase`
- missing values become `unknown`

These grouped columns stay categorical and will later be passed directly to native `CatBoost`.

In [5]:
def map_emp_title_to_group(emp_title: str) -> str:
    if pd.isna(emp_title):
        return "unknown"
    title = str(emp_title).strip().lower()
    if not title or title in {"nan", "none"}:
        return "unknown"

    keyword_groups = [
        ("retired", ["retired"]),
        ("student", ["student", "intern"]),
        ("self_employed_business_owner", ["self employed", "self-employed", "owner", "founder", "co-owner", "entrepreneur", "small business", "business owner"]),
        ("government_public_safety", ["government", "city of", "county", "state of", "police", "officer", "sheriff", "fire", "army", "navy", "air force", "marine", "military", "federal", "usps", "postal"]),
        ("healthcare", ["nurse", "rn", "doctor", "physician", "medical", "hospital", "health", "dental", "dentist", "pharmacy", "therap", "clinical", "caregiver"]),
        ("education", ["teacher", "professor", "school", "educat", "principal", "instructor", "faculty", "counselor"]),
        ("finance_accounting", ["account", "finance", "financial", "bank", "banker", "credit", "loan", "underwriter", "analyst", "controller", "auditor", "bookkeeper"]),
        ("engineering_it", ["engineer", "developer", "programmer", "software", "network", "systems", "architect", "cyber", "database", "data scientist", "it support", "help desk"]),
        ("management_administration", ["manager", "management", "director", "supervisor", "administrator", "admin", "coordinator", "executive", "operations", "project manager"]),
        ("sales_marketing_real_estate", ["sales", "marketing", "account executive", "realtor", "real estate", "broker", "leasing"]),
        ("transportation_logistics", ["driver", "truck", "transport", "logistics", "warehouse", "delivery", "shipping", "forklift", "dispatcher"]),
        ("construction_skilled_trade", ["construction", "electric", "plumb", "carpent", "mechanic", "machinist", "welder", "contractor", "installer", "maintenance", "hvac"]),
        ("manufacturing_production", ["manufacturing", "production", "assembler", "operator", "factory", "plant"]),
        ("service_retail_hospitality", ["retail", "cashier", "server", "waiter", "waitress", "bartender", "cook", "chef", "restaurant", "hotel", "customer service", "store"]),
    ]

    for group_name, keywords in keyword_groups:
        for keyword in keywords:
            if keyword in title:
                return group_name
    return "other"

def map_purpose_to_group(purpose: str) -> str:
    if pd.isna(purpose):
        return "unknown"
    value = str(purpose).strip().lower()
    if not value or value in {"nan", "none"}:
        return "unknown"

    if value in {"credit_card", "debt_consolidation"}:
        return "debt_refinancing"
    if value in {"home_improvement", "house", "moving"}:
        return "housing_related"
    if value in {"car", "major_purchase", "vacation", "wedding"}:
        return "major_or_discretionary_purchase"
    if value == "medical":
        return "medical_emergency"
    if value == "small_business":
        return "business_investment"
    if value == "educational":
        return "education"
    if value == "renewable_energy":
        return "energy_or_special_project"
    if value == "other":
        return "other"
    return value

grouped_df = engineered_df.copy()
grouped_df["occupation_group"] = source_df["emp_title"].apply(map_emp_title_to_group).astype(str)
grouped_df["purpose_group"] = source_df["purpose"].apply(map_purpose_to_group).astype(str)

print("grouped_df shape:", grouped_df.shape)
display(grouped_df["occupation_group"].value_counts().head(10).rename("count").to_frame())
display(grouped_df["purpose_group"].value_counts().rename("count").to_frame())
grouped_df[["occupation_group", "purpose_group"]].head()


grouped_df shape: (87496, 81)


,count
occupation_group,
other,25278
management_administration,21053
finance_accounting,6575
healthcare,6034
unknown,5598
education,3998
engineering_it,3874
transportation_logistics,2765
sales_marketing_real_estate,2585


,count
purpose_group,
debt_refinancing,71333
housing_related,6378
other,4883
major_or_discretionary_purchase,3036
medical_emergency,1035
business_investment,777
energy_or_special_project,54


,occupation_group,purpose_group
0,other,debt_refinancing
1,engineering_it,business_investment
2,other,major_or_discretionary_purchase
3,other,debt_refinancing
4,management_administration,debt_refinancing


## Box 6: Handle Missing Values And Build The Final CatBoost Frame

This box converts the grouped dataset into the final feature frame for the clean native `CatBoost` model.

How missing values are handled:
- for sparse `months since` and related utilization/limit fields, create a matching `_is_nan` indicator, then fill the original value with `0`
- for selected sparse count/balance columns where missing is treated as absence, fill with `0`
- for a few remaining light-missing numeric columns, also fill with `0`
- categorical columns are filled with `unknown` and kept as strings

This box also removes the remaining non-feature columns and defines the native categorical column list for `CatBoost`.

In [6]:
MONTHS_SINCE_COLUMNS = [
    "mths_since_last_delinq", "mths_since_last_record", "mths_since_last_major_derog",
    "mths_since_rcnt_il", "mo_sin_old_il_acct", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl",
    "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_revol_delinq",
    "num_bc_sats", "num_sats", "all_util", "bc_util", "percent_bc_gt_75", "revol_util",
    "il_util", "tot_cur_bal", "mo_sin_old_rev_tl_op", "mths_since_recent_inq",
    "pct_tl_nvr_dlq", "tot_hi_cred_lim", "total_bc_limit", "num_tl_120dpd_2m"
]

SAFE_TO_FILL_ZERO = [
    "open_acc_6m", "open_rv_12m", "open_rv_24m", "total_bal_il", "max_bal_bc", "inq_fi",
    "inq_last_12m", "total_cu_tl", "acc_open_past_24mths", "mort_acc",
    "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_tl",
    "num_il_tl", "num_op_rev_tl", "open_act_il", "open_il_12m", "open_il_24m",
    "tot_coll_amt", "num_tl_30dpd", "num_tl_90g_dpd_24m", "total_il_high_credit_limit"
]

FINAL_LIGHT_MISSING_FILL = [
    "annual_inc", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "total_acc", "collections_12_mths_ex_med", "acc_now_delinq", "total_rev_hi_lim",
    "avg_cur_bal", "chargeoff_within_12_mths", "delinq_amnt", "tax_liens",
    "pub_rec_bankruptcies", "tot_cur_bal", "total_bc_limit", "tot_hi_cred_lim"
]

CATBOOST_CATEGORICAL_COLUMNS = [
    "home_ownership", "verification_status", "emp_length_cat", "occupation_group", "purpose_group"
]

final_df = grouped_df.copy()

for column in CATBOOST_CATEGORICAL_COLUMNS:
    if column in final_df.columns:
        final_df[column] = final_df[column].fillna("unknown").astype(str)

for column in MONTHS_SINCE_COLUMNS:
    if column in final_df.columns:
        final_df[f"{column}_is_nan"] = final_df[column].isna().astype(int)
        final_df[column] = final_df[column].fillna(0)

for column in SAFE_TO_FILL_ZERO:
    if column in final_df.columns:
        final_df[column] = final_df[column].fillna(0)

for column in FINAL_LIGHT_MISSING_FILL:
    if column in final_df.columns:
        final_df[column] = final_df[column].fillna(0)

FINAL_NON_FEATURE_COLUMNS = [
    "target", "loan_status", "application_type", "last_fico_range_high", "last_fico_range_low"
]

X_final = final_df.drop(columns=[column for column in FINAL_NON_FEATURE_COLUMNS if column in final_df.columns]).copy()
y_final = final_df["target"].astype(int).copy()

for column in X_final.columns:
    if column not in CATBOOST_CATEGORICAL_COLUMNS:
        X_final[column] = pd.to_numeric(X_final[column], errors="coerce").fillna(0)

catboost_cat_features = [column for column in CATBOOST_CATEGORICAL_COLUMNS if column in X_final.columns]

print("X_final shape:", X_final.shape)
print("y_final shape:", y_final.shape)
print("CatBoost categorical columns:", catboost_cat_features)
display(pd.DataFrame({"categorical_feature": catboost_cat_features}))
X_final.head()


X_final shape: (87496, 100)
y_final shape: (87496,)
CatBoost categorical columns: ['home_ownership', 'verification_status', 'emp_length_cat', 'occupation_group', 'purpose_group']


,categorical_feature
0,home_ownership
1,verification_status
2,emp_length_cat
3,occupation_group
4,purpose_group


,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,...,percent_bc_gt_75_is_nan,revol_util_is_nan,il_util_is_nan,tot_cur_bal_is_nan,mo_sin_old_rev_tl_op_is_nan,mths_since_recent_inq_is_nan,pct_tl_nvr_dlq_is_nan,tot_hi_cred_lim_is_nan,total_bc_limit_is_nan,num_tl_120dpd_2m_is_nan
0,3600.0,36,10.0,MORTGAGE,55000.0,Not Verified,5.91,0.0,1.0,30.0,...,0,0,0,0,0,0,0,0,0,0
1,24700.0,36,10.0,MORTGAGE,65000.0,Not Verified,16.06,1.0,4.0,6.0,...,0,0,0,0,0,0,0,0,0,0
2,10400.0,60,3.0,MORTGAGE,104433.0,Source Verified,25.37,1.0,3.0,12.0,...,0,0,0,0,0,0,0,0,0,0
3,11950.0,36,4.0,RENT,34000.0,Source Verified,10.20,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,0
4,20000.0,36,10.0,MORTGAGE,180000.0,Not Verified,14.67,0.0,0.0,49.0,...,0,0,0,0,0,0,0,0,0,0


## Box 7: Define The Four Final Feature Subsets

The final clean pipeline does not train just one model. It evaluates the same four final native `CatBoost` variants that were selected in the experimentation notebook:
- `top_10_correlation`
- `top_20_combined`
- `top_40_importance`
- `full_native_grouped`

These are hardcoded from the final experimentation results, so this clean notebook does not depend on reading ranking files.

How they were originally chosen:
- `top_10_correlation`: the final top-10 subset selected from the correlation ranking.
- `top_20_combined`: the final top-20 subset selected from the combined correlation-plus-importance ranking.
- `top_40_importance`: the final top-40 subset selected from the logistic-importance ranking.
- `full_native_grouped`: the full clean native feature set.

This box also removes raw `emp_length` so the clean pipeline matches the final experimentation setup that kept `emp_length_cat` but not numeric `emp_length`.

In [7]:
X_native = X_final.drop(columns=["emp_length"], errors="ignore").copy()
catboost_cat_features = [column for column in catboost_cat_features if column in X_native.columns]

final_feature_sets = {
    "top_10_correlation": [
        "term", "acc_open_past_24mths", "dti", "verification_status", "num_tl_op_past_12m",
        "open_rv_24m", "num_actv_rev_tl", "num_rev_tl_bal_gt_0", "percent_bc_gt_75",
        "avg_cur_bal", "home_ownership", "emp_length_cat", "occupation_group", "purpose_group",
    ],
    "top_20_combined": [
        "term", "dti", "acc_open_past_24mths", "num_actv_rev_tl", "percent_bc_gt_75",
        "num_rev_tl_bal_gt_0", "loan_amnt", "total_bc_limit", "all_util_is_nan", "all_util",
        "open_rv_24m", "open_rv_12m", "mths_since_rcnt_il_is_nan", "open_acc_6m", "il_util",
        "revol_util", "mort_acc", "num_actv_bc_tl", "bc_util", "tot_cur_bal",
        "home_ownership", "verification_status", "emp_length_cat", "occupation_group", "purpose_group",
    ],
    "top_40_importance": [
        "bc_util_is_nan", "percent_bc_gt_75_is_nan", "all_util_is_nan", "mths_since_rcnt_il_is_nan",
        "pct_tl_nvr_dlq_is_nan", "term", "total_il_high_credit_limit", "dti", "total_bal_ex_mort",
        "acc_open_past_24mths", "loan_amnt", "total_bc_limit", "percent_bc_gt_75", "num_actv_rev_tl",
        "num_rev_tl_bal_gt_0", "total_acc", "revol_bal", "open_rv_12m", "mo_sin_old_rev_tl_op",
        "emp_length_cat", "all_util", "il_util", "num_actv_bc_tl", "revol_util", "open_acc_6m",
        "credit_maturity", "mths_since_last_delinq_is_nan", "open_rv_24m", "mort_acc",
        "mo_sin_rcnt_rev_tl_op_is_nan", "mo_sin_rcnt_tl_is_nan", "tot_cur_bal_is_nan",
        "mo_sin_old_rev_tl_op_is_nan", "tot_hi_cred_lim_is_nan", "tot_cur_bal", "pct_tl_nvr_dlq",
        "mths_since_recent_inq_is_nan", "mths_since_last_record", "delinq_2yrs", "bc_util",
        "home_ownership", "verification_status", "occupation_group", "purpose_group",
    ],
    "full_native_grouped": [
        "loan_amnt", "term", "home_ownership", "annual_inc", "verification_status", "dti",
        "delinq_2yrs", "inq_last_6mths", "mths_since_last_delinq", "mths_since_last_record",
        "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc", "collections_12_mths_ex_med",
        "mths_since_last_major_derog", "acc_now_delinq", "tot_coll_amt", "tot_cur_bal", "open_acc_6m",
        "open_act_il", "open_il_12m", "open_il_24m", "mths_since_rcnt_il", "total_bal_il", "il_util",
        "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util", "total_rev_hi_lim", "inq_fi",
        "total_cu_tl", "inq_last_12m", "acc_open_past_24mths", "avg_cur_bal", "bc_util",
        "chargeoff_within_12_mths", "delinq_amnt", "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op",
        "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl", "mort_acc", "mths_since_recent_bc",
        "mths_since_recent_bc_dlq", "mths_since_recent_inq", "mths_since_recent_revol_delinq",
        "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_sats", "num_bc_tl",
        "num_il_tl", "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats",
        "num_tl_120dpd_2m", "num_tl_30dpd", "num_tl_90g_dpd_24m", "num_tl_op_past_12m",
        "pct_tl_nvr_dlq", "percent_bc_gt_75", "pub_rec_bankruptcies", "tax_liens", "tot_hi_cred_lim",
        "total_bal_ex_mort", "total_bc_limit", "total_il_high_credit_limit", "credit_maturity",
        "emp_length_cat", "occupation_group", "purpose_group", "mths_since_last_delinq_is_nan",
        "mths_since_last_record_is_nan", "mths_since_last_major_derog_is_nan", "mths_since_rcnt_il_is_nan",
        "mo_sin_old_il_acct_is_nan", "mo_sin_rcnt_rev_tl_op_is_nan", "mo_sin_rcnt_tl_is_nan",
        "mths_since_recent_bc_is_nan", "mths_since_recent_bc_dlq_is_nan", "mths_since_recent_revol_delinq_is_nan",
        "num_bc_sats_is_nan", "num_sats_is_nan", "all_util_is_nan", "bc_util_is_nan",
        "percent_bc_gt_75_is_nan", "revol_util_is_nan", "il_util_is_nan", "tot_cur_bal_is_nan",
        "mo_sin_old_rev_tl_op_is_nan", "mths_since_recent_inq_is_nan", "pct_tl_nvr_dlq_is_nan",
        "tot_hi_cred_lim_is_nan", "total_bc_limit_is_nan", "num_tl_120dpd_2m_is_nan",
    ],
}

missing_features = {
    subset_name: [column for column in columns if column not in X_native.columns]
    for subset_name, columns in final_feature_sets.items()
}
missing_features = {key: value for key, value in missing_features.items() if value}
if missing_features:
    raise ValueError(f"Some hardcoded final features are missing from X_native: {missing_features}")

final_feature_set_summary_df = pd.DataFrame([
    {
        "subset_name": subset_name,
        "feature_count": len(columns),
        "categorical_feature_count": sum(column in catboost_cat_features for column in columns),
        "categorical_features": ", ".join([column for column in columns if column in catboost_cat_features]),
        "example_features": " | ".join(columns[:10]),
    }
    for subset_name, columns in final_feature_sets.items()
])

print("X_native shape:", X_native.shape)
display(final_feature_set_summary_df)


X_native shape: (87496, 99)


,subset_name,feature_count,categorical_feature_count,categorical_features,example_features
0,top_10_correlation,14,5,"verification_status, home_ownership, emp_lengt...",term | acc_open_past_24mths | dti | verificati...
1,top_20_combined,25,5,"home_ownership, verification_status, emp_lengt...",term | dti | acc_open_past_24mths | num_actv_r...
2,top_40_importance,44,5,"emp_length_cat, home_ownership, verification_s...",bc_util_is_nan | percent_bc_gt_75_is_nan | all...
3,full_native_grouped,99,5,"home_ownership, verification_status, emp_lengt...",loan_amnt | term | home_ownership | annual_inc...


## Box 8: Train And Evaluate The Four Final Native CatBoost Models

This box trains and evaluates the four final clean models.

To stay consistent with the earlier final clean-model experiments, we first build a balanced working dataset from the current debug subset:
- keep all available `Charged Off` rows in the current subset
- randomly sample the same number of `Fully Paid` rows

Then each of the four feature subsets is evaluated with:
- native `CatBoost`
- `3`-fold stratified cross-validation
- Accuracy, Precision, Recall, and Weighted F1


In [8]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold

positive_indices = y_final[y_final == 1].index
negative_indices = y_final[y_final == 0].index
sampled_negative_indices = y_final[y_final == 0].sample(n=len(positive_indices), random_state=42).index
balanced_indices = positive_indices.union(sampled_negative_indices)

X_balanced = X_native.loc[balanced_indices].reset_index(drop=True)
y_balanced = y_final.loc[balanced_indices].reset_index(drop=True)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
final_model_summary_rows = []
final_model_fold_rows = []

for subset_name in ["top_10_correlation", "top_20_combined", "top_40_importance", "full_native_grouped"]:
    subset_columns = final_feature_sets[subset_name]
    X_subset = X_balanced[subset_columns].copy()
    subset_cat_columns = [column for column in catboost_cat_features if column in subset_columns]
    fit_times = []
    fold_metrics = []

    for fold_index, (train_idx, test_idx) in enumerate(cv.split(X_subset, y_balanced), start=1):
        X_train = X_subset.iloc[train_idx].copy()
        X_test = X_subset.iloc[test_idx].copy()
        y_train = y_balanced.iloc[train_idx]
        y_test = y_balanced.iloc[test_idx]

        fold_model = CatBoostClassifier(
            depth=6,
            iterations=500,
            learning_rate=0.05,
            l2_leaf_reg=3,
            loss_function="Logloss",
            verbose=0,
            random_state=42,
            cat_features=subset_cat_columns,
        )
        fold_start = pd.Timestamp.now()
        fold_model.fit(X_train, y_train)
        fit_times.append((pd.Timestamp.now() - fold_start).total_seconds())
        fold_predictions = pd.Series(fold_model.predict(X_test).reshape(-1)).astype(int)

        fold_metric = {
            "accuracy": float(accuracy_score(y_test, fold_predictions)),
            "precision": float(precision_score(y_test, fold_predictions, zero_division=0)),
            "recall": float(recall_score(y_test, fold_predictions, zero_division=0)),
            "weighted_f1": float(f1_score(y_test, fold_predictions, average="weighted", zero_division=0)),
        }
        fold_metrics.append(fold_metric)
        final_model_fold_rows.append({
            "subset_name": subset_name,
            "feature_count": int(len(subset_columns)),
            "categorical_feature_count": int(len(subset_cat_columns)),
            "fold": int(fold_index),
            **fold_metric,
        })

    final_model_summary_rows.append({
        "subset_name": subset_name,
        "feature_count": int(len(subset_columns)),
        "categorical_feature_count": int(len(subset_cat_columns)),
        "categorical_features": ", ".join(subset_cat_columns),
        "accuracy_mean": float(pd.DataFrame(fold_metrics)["accuracy"].mean()),
        "precision_mean": float(pd.DataFrame(fold_metrics)["precision"].mean()),
        "recall_mean": float(pd.DataFrame(fold_metrics)["recall"].mean()),
        "weighted_f1_mean": float(pd.DataFrame(fold_metrics)["weighted_f1"].mean()),
        "fit_time_mean_sec": float(pd.Series(fit_times).mean()),
    })

final_model_summary_df = pd.DataFrame(final_model_summary_rows).sort_values(
    ["weighted_f1_mean", "recall_mean", "precision_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)
final_model_folds_df = pd.DataFrame(final_model_fold_rows)

print("X_balanced shape:", X_balanced.shape)
print("y_balanced value counts:")
print(y_balanced.value_counts().sort_index())
display(final_model_summary_df)


X_balanced shape: (35012, 99)
y_balanced value counts:
target
0    17506
1    17506
Name: count, dtype: int64


,subset_name,feature_count,categorical_feature_count,categorical_features,accuracy_mean,precision_mean,recall_mean,weighted_f1_mean,fit_time_mean_sec
0,full_native_grouped,99,5,"home_ownership, verification_status, emp_lengt...",0.674626,0.680630,0.658003,0.674533,22.247855
1,top_40_importance,44,5,"home_ownership, verification_status, emp_lengt...",0.669628,0.675809,0.652063,0.669513,21.023148
2,top_20_combined,25,5,"home_ownership, verification_status, emp_lengt...",0.666886,0.674940,0.644008,0.666697,20.985297
3,top_10_correlation,14,5,"home_ownership, verification_status, emp_lengt...",0.658574,0.668743,0.628471,0.658254,19.438374
